# Quick CUDA Test - CFD Solver

Fast compilation and testing of specific CUDA kernels.

---

In [ ]:
# Quick setup
!pip install -q ipywidgets
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import os

# Clone if not exists
if not os.path.exists('CFD_Solver_withCUDA'):
    !git clone https://github.com/rameshkolluru43/CFD_Solver_withCUDA.git
%cd CFD_Solver_withCUDA

## Test Individual Kernel

In [ ]:
# Select kernel to test
import ipywidgets as widgets
from IPython.display import display

kernels = [
    "Flux_Calculations_Cuda_Kernels.cu",
    "Roe_Flux_Cuda_Kernels.cu",
    "HLLC_LLF_Flux_Cuda_Kernels.cu",
    "MUSCL_WENO_Reconstruction_Cuda_Kernels.cu",
    "Boundary_Conditions_Cuda_Kernels.cu",
    "Time_Integration_Cuda_Kernels.cu",
    "Gradient_Calculation_Cuda_Kernels.cu",
    "Limiter_Cuda_Kernels.cu",
    "Viscous_Flux_Cuda_Kernels.cu"
]

kernel_dropdown = widgets.Dropdown(
    options=kernels,
    description='Kernel:',
    disabled=False,
)

display(kernel_dropdown)

In [ ]:
# Compile selected kernel
selected_kernel = kernel_dropdown.value
kernel_path = f"CUDA_KERNELS/{selected_kernel}"

print(f"🔧 Compiling: {selected_kernel}\n")
print("="*60)

!nvcc -c {kernel_path} \
    -I./include \
    -I./CUDA_KERNELS \
    -arch=sm_75 \
    -std=c++17 \
    --expt-relaxed-constexpr \
    --ptxas-options=-v \
    -o /tmp/{selected_kernel}.o

print("\n" + "="*60)

if os.path.exists(f"/tmp/{selected_kernel}.o"):
    print(f"✅ {selected_kernel} compiled successfully!")
    !ls -lh /tmp/{selected_kernel}.o
else:
    print(f"❌ Compilation failed for {selected_kernel}")

## View Kernel Source

In [ ]:
# Display kernel source code
print(f"📄 Source: {selected_kernel}\n")
print("="*60)

!head -100 CUDA_KERNELS/{selected_kernel}

print("\n... (showing first 100 lines)")
print("\nTotal lines:")
!wc -l CUDA_KERNELS/{selected_kernel}

## Analyze Kernel

In [ ]:
# Extract kernel function names
print(f"🔍 Analyzing {selected_kernel}...\n")

print("Global Kernels (__global__):")
!grep -n "__global__" CUDA_KERNELS/{selected_kernel}

print("\nDevice Functions (__device__):")
!grep -n "__device__" CUDA_KERNELS/{selected_kernel} | head -20

print("\nShared Memory Usage:")
!grep -n "__shared__" CUDA_KERNELS/{selected_kernel}

print("\nTemplate Kernels:")
!grep -n "template" CUDA_KERNELS/{selected_kernel}

## Build Full Project

In [ ]:
# Install dependencies
!apt-get update -qq && apt-get install -y cmake libjsoncpp-dev -qq

# Configure and build
!cp CMakeLists_Colab.txt CMakeLists.txt
!mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release && make -j4 CFD_solver_gpu

## GPU Info

In [ ]:
!nvidia-smi
print("\n" + "="*60)
!nvcc --version